In [1]:
import venv, os, subprocess, sys

VENV_DIR = os.path.join(os.getcwd(), "sls-mkt_rag_venv")
venv.create(VENV_DIR, with_pip=True)

PYTHON = os.path.join(VENV_DIR, "Scripts", "python.exe")
print(f"Venv Python: {PYTHON}")
print(f"Kernel Python: {sys.executable}")  # Will differ — that's expected

Venv Python: c:\Git projects\sales_marketing_rag\sls-mkt_rag_venv\Scripts\python.exe
Kernel Python: c:\Users\brand\AppData\Local\Programs\Python\Python311\python.exe


In [8]:
import os
import nbformat

dirs = [
    "notebooks",
    "docker"
]
# 
notebooks = [
   "notebooks/01_rfm_engineering.ipynb", #3
    "notebooks/02_clv_prediction.ipynb", #3
    "notebooks/03_segmentation_pgvector.ipynb", #3
    "notebooks/04_similarity_search.ipynb", #3
    "notebooks/05_business_interpretation.ipynb", #3
    "docker/docker-compose.yml", #3
    "requirements.txt",
    ".gitattributes",
    "README.md"
]

for folder in dirs:
    os.makedirs(folder, exist_ok=True)

for nb_path in notebooks:
    nb = nbformat.v4.new_notebook()
    with open(nb_path, "w") as f:
        nbformat.write(nb, f)
        

ModuleNotFoundError: No module named 'nbformat'

In [3]:
import os, psycopg
from pgvector.psycopg import register_vector
from dotenv import load_dotenv

load_dotenv(r"C:\Git projects\sales_marketing_rag\.env")
 
DIM = int(os.environ["EMBED_DIM"])
conn = psycopg.connect(os.environ["DATABASE_URL"], autocommit=True)

# 1. Create the extension first
conn.execute("CREATE EXTENSION IF NOT EXISTS vector;")

# 2. Register the vector type AFTER the extension is guaranteed to exist
register_vector(conn)
 
# 3. Create your relational tables
conn.execute(f"""
CREATE TABLE IF NOT EXISTS documents (
    id          BIGSERIAL PRIMARY KEY,
    source      TEXT NOT NULL,          -- e.g. 'Q1-2026 earnings call'
    doc_type    TEXT NOT NULL,          -- 'transcript' | '10k'
    created_at  TIMESTAMPTZ DEFAULT now()
);
 
CREATE TABLE IF NOT EXISTS chunks (
    id          BIGSERIAL PRIMARY KEY,
    document_id BIGINT REFERENCES documents(id) ON DELETE CASCADE,
    chunk_index INT NOT NULL,
    content     TEXT NOT NULL,
    token_count INT,
    embedding   vector({DIM}),
    tsv         tsvector GENERATED ALWAYS AS (to_tsvector('english', content)) STORED
);
""")
print("schema ready")


ValueError: invalid literal for int() with base 10: '1536                # 1536 for 3-small, 768 for bge-base'

In [4]:
import psycopg
from rag.config import DATABASE_URL
print(DATABASE_URL)          # sanity-check it actually says 5433
with psycopg.connect(DATABASE_URL) as conn:
    print("connected")

ModuleNotFoundError: No module named 'rag'